# Credit Card Fraud — extreme class imbalance, honest PR-AUC

Case: [Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)
(284,807 European card transactions over two days, 492 frauds — **0.172%**
positives; 28 anonymized PCA components + `Time` + `Amount`).

Why this case matters for an honesty-focused library:

- with 0.17% positives, accuracy and even ROC AUC look deceptively great;
  selection is driven by **`metric="pr_auc"`** (average precision), the metric
  the dataset's own documentation recommends;
- fraud scores feed downstream decisions, so probabilities should be
  trustworthy: `calibrate="auto"` adds leakage-free cross-fit calibration;
- a 20% stratified outer holdout (~98 frauds) is scored exactly once.

Heads-up: this is the heavyweight notebook — the full zoo on 284k rows takes
a few minutes on CPU (~7 min on a desktop). (`AutoML(budget=...)` can cap it
gracefully if you just want a quick pass.)

In [1]:
import json
import logging
import os
import shutil
import subprocess
import time
from pathlib import Path

import pandas as pd
from IPython.display import Markdown

from honestml import AutoML, CVConfig, render_report, save_run_report

SEED = 42
DATA = Path("data/creditcardfraud")
RESULTS = Path("results/creditcardfraud")
RESULTS.mkdir(parents=True, exist_ok=True)
KAGGLE = os.environ.get("KAGGLE_BIN") or shutil.which("kaggle") or str(Path.home() / ".local" / "bin" / "kaggle")

logging.basicConfig(format="%(levelname)s %(name)s: %(message)s")
logging.getLogger("honestml").setLevel(logging.INFO)

In [2]:
if not (DATA / "creditcard.csv").exists():
    DATA.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [KAGGLE, "datasets", "download", "-d", "mlg-ulb/creditcardfraud", "-p", str(DATA), "--unzip"],
        check=True,
    )

In [3]:
df = pd.read_csv(DATA / "creditcard.csv")
y = df.pop("Class")
X = df
print(f"rows: {len(X)}, frauds: {y.sum()} ({y.mean():.5%})")
X.head()

rows: 284807, frauds: 492 (0.17275%)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,0.251412,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.069083,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.524980,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.208038,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,0.408542,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99


## Fit

In [4]:
model = AutoML(
    task="binary",
    metric="pr_auc",
    cv=CVConfig(outer_holdout=0.2, calibrate="auto"),
    random_state=SEED,
)
t0 = time.perf_counter()
model.fit(X, y)
fit_seconds = time.perf_counter() - t0
print(f"fit took {fit_seconds / 60:.1f} min")

INFO honestml: stage key=run stage=selection elapsed=245.8s


WARNING honestml.adapters.boosting: boosting 'catboost' trained without early stopping; leaderboard comparison may favor overfit settings


INFO honestml: stage key=run stage=refit elapsed=3.6s


INFO honestml: stage key=run stage=refit elapsed=4.3s


INFO honestml: stage key=run stage=finalize elapsed=4.3s


fit took 4.2 min


In [5]:
report = model.run_report_
pd.DataFrame(report["leaderboard"])

,model_id,score,rank
0,catboost,0.840848,1
1,lightgbm,0.780965,2
2,linear,0.761127,3
3,xgboost,0.709970,4
4,baseline,0.001726,5


Every gradient boosting learns to rank the rare class. CatBoost leads on
honest OOF PR-AUC (0.84), with LightGBM (0.78) and XGBoost (0.71) behind, and
a plain linear model (0.76) holding its own — the median imputer keeps it and
the stratified `baseline` (PR-AUC ≈ prevalence, 0.002) in the comparison
rather than letting NaN-handling quietly evict the non-boosting candidates.
The point holds even without a dramatic blow-up: on 0.17% positives ROC AUC
flatters everything, so the only ranking you can trust is the one computed in
the metric you actually care about — honest out-of-fold PR-AUC — which is
exactly what separates CatBoost's 0.84 from XGBoost's 0.71 here.

## The honesty check

This is exactly the setting where leaderboard optimism bites hardest: with 492
positives, out-of-fold estimates are noisy and the temptation to trust the
best CV number is maximal. The untouched holdout is the reality check.

In [6]:
selection = next(e["score"] for e in report["leaderboard"] if e["model_id"] == report["winner"])
print(f"winner          : {report['winner']}")
print(f"equivalence band: {report['band']['member_ids']}")
print(f"selection score : {selection:.4f}   (out-of-fold PR-AUC)")
print(f"holdout score   : {report['holdout_score']:.4f}   (untouched 20%, scored once)")
print(f"optimism        : {selection - report['holdout_score']:+.4f}")

winner          : catboost
equivalence band: ['catboost']
selection score : 0.8408   (out-of-fold PR-AUC)
holdout score   : 0.8825   (untouched 20%, scored once)
optimism        : -0.0417


In [7]:
save_run_report(report, RESULTS)
md_path = render_report(report, RESULTS, fmt="md")
Markdown(md_path.read_text(encoding="utf-8"))

# AutoML run report

## Run

| | |
|---|---|
| task | binary |
| metric | pr_auc |
| winner | catboost |
| holdout_score | 0.882525 |
| honestml_version | 1.0.0 |
| run_fingerprint | c1e18be05210b7ceda566743a633b56b7cfee5f45de8053a9df9e25c20d2b42b |
| significance | bootstrap |
| preset | n/a |

## Equivalence band

| | |
|---|---|
| members | catboost |
| width | 1 |
| unstable | False |
| winner_by_tiebreak | False |

## Budget

| | |
|---|---|
| mode | none |
| exhausted | False |
| exhausted_by | n/a |
| skipped | n/a |

## Serving

| | |
|---|---|
| finalize | True |
| shipped_on | all |
| outer_holdout | 0.2 |

## Leaderboard

| rank | model | score |
|---|---|---|
| 1 | catboost (winner) | 0.840848 |
| 2 | lightgbm | 0.780965 |
| 3 | linear | 0.761127 |
| 4 | xgboost | 0.70997 |
| 5 | baseline | 0.00172577 |

## Timings (s)

| stage | elapsed |
|---|---|
| run.selection | 245.8 |
| run.refit | 4.3 |
| run.finalize | 4.3 |

## Resolved config

```json
{
  "seed": 42,
  "cv": {
    "scheme": "stratified",
    "n_splits": 5,
    "n_test": 1,
    "n_es": 1,
    "purge": 0,
    "embargo": 0,
    "period": null,
    "period_size": null,
    "step_periods": null,
    "purge_delta": null,
    "embargo_delta": null,
    "max_train_periods": null,
    "max_train_size": null,
    "weighting": "pooled",
    "calibrate": "auto",
    "selection": "raw",
    "refinement_min_oof": 2000,
    "outer_holdout": 0.2
  },
  "budget": {
    "mode": "none",
    "time_budget_s": null,
    "n_trials": null,
    "memory_limit_mb": null
  },
  "fe": {
    "target_encoding": false,
    "te_smoothing": 10.0,
    "frequency_encoding": false,
    "intersections": false,
    "max_pairs": 50
  },
  "fs": null,
  "hpo": null,
  "ensemble": null,
  "significance": "bootstrap",
  "run_mode": "full",
  "model_types": [
    "catboost",
    "lightgbm"
  ]
}
```


## Comparison with published results

[Andrade-Arenas & Yactayo-Arias (IJSSE 15(5), 2025)](https://doi.org/10.18280/ijsse.150504)
benchmark seven models on this exact dataset with an 80/20 split and SMOTE
applied to the training side:

| Model (their study) | AUPRC | ROC AUC |
| --- | --- | --- |
| Random Forest | 0.871 | 0.978 |
| XGBoost | 0.867 | 0.983 |
| AdaBoost | 0.808 | 0.927 |
| Logistic Regression | 0.724 | 0.971 |

Two things to read off that table together with our run:

1. **The imbalance story.** ROC AUC ≈ 0.97–0.98 looks stellar even for models
   whose PR-AUC differs by 15 points — with 0.17% positives the ROC curve
   barely feels false positives. That is why this notebook selects on
   `pr_auc` directly; an AutoML ranking candidates by ROC AUC here would be
   optimizing a mirage.
2. **The honesty story.** Their split is a single 80/20 split scored on the
   same test set the paper reports — our PR-AUC above comes from an untouched
   holdout that took no part in any decision, next to the out-of-fold
   selection score and their difference (the optimism). Same data, same
   family of models; the difference is how much you can trust the number.

## Level 2: HPO + feature selection (no FE — the features are PCA components)

What changes against level 1:

- `preset="best"` + an explicit `HPOConfig(n_trials=20, timeout_s=900)` —
  Optuna tuning capped at 20 trials / 15 minutes per model (284k rows). Level 1
  already puts every boosting in a healthy range, so this is not an imbalance
  rescue — it just asks whether tuned hyperparameters can edge out the
  (early-stopped) defaults;
- `FeatureSelectionConfig(strategy="importance", cutoff="auto")` — how many
  of the 28 anonymous PCA components carry fraud signal, with an honest
  no-selection gate that ships all features when the trimmed subset isn't
  demonstrably better;
- feature engineering stays off: target/frequency encoding and intersections
  need categorical columns, and this dataset has none by construction;
- calibration and the untouched 20% holdout as in level 1.

In [8]:
from honestml import FeatureSelectionConfig, HPOConfig

model_full = AutoML(
    task="binary",
    metric="pr_auc",
    preset="best",
    hpo=HPOConfig(n_trials=20, timeout_s=900),
    feature_selection=FeatureSelectionConfig(strategy="importance", cutoff="auto"),
    cv=CVConfig(outer_holdout=0.2, calibrate="auto"),
    random_state=SEED,
)
t0 = time.perf_counter()
model_full.fit(X, y)
print(f"level-2 fit took {(time.perf_counter() - t0) / 60:.1f} min")

INFO honestml.composition.presets: preset best applied: ['ensemble']


C:\Users\Admin\Documents\AutoML\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


WARNING honestml.adapters.boosting: boosting 'lightgbm' trained without early stopping; leaderboard comparison may favor overfit settings


WARNING honestml.adapters.boosting: boosting 'xgboost' trained without early stopping; leaderboard comparison may favor overfit settings


INFO honestml: stage key=run stage=hpo elapsed=483.9s


...

INFO honestml: stage key=run stage=ensemble elapsed=164.5s


INFO honestml: stage key=run stage=refit elapsed=2.0s


INFO honestml: stage key=run stage=refit elapsed=2.5s


INFO honestml: stage key=run stage=finalize elapsed=2.5s


level-2 fit took 17.9 min


In [9]:
report_full = model_full.run_report_
sel_full = next(e["score"] for e in report_full["leaderboard"] if e["model_id"] == report_full["winner"])
print(f"winner          : {report_full['winner']}")
print(f"selection score : {sel_full:.4f}   (level 1: {selection:.4f})")
print(f"holdout score   : {report_full['holdout_score']:.4f}   (level 1: {report['holdout_score']:.4f})")
print(f"optimism        : {sel_full - report_full['holdout_score']:+.4f}")
print()
print("preset  :", report_full["preset"])
print("hpo     :", json.dumps(report_full["hpo"], default=str)[:600])
print("fs      :", json.dumps(report_full["feature_selection"], default=str)[:600])
print("ensemble:", json.dumps(report_full["ensemble"], default=str)[:300])
pd.DataFrame(report_full["leaderboard"])

winner          : xgboost
selection score : 0.8436   (level 1: 0.8408)
holdout score   : 0.8714   (level 1: 0.8825)
optimism        : -0.0278

preset  : {'name': 'best', 'applied': ['ensemble']}
hpo     : {"backend": "optuna", "inner_cv": 3, "deterministic": false, "selection_oof_is_post_tuning": true, "tuned_on_full_feature_space": true, "cost_estimate_fits": 180, "tuned": {"catboost": {"chosen_params": {"depth": 8, "learning_rate": 0.029676178173412764, "iterations": 350, "l2_leaf_reg": 1.3106323224887602, "subsample": 0.7974844936961976, "one_hot_max_size": 26}, "inner_best_score": 0.845644255263431, "n_trials_run": 20}, "lightgbm": {"chosen_params": {"max_depth": 9, "learning_rate": 0.039631425462889405, "n_estimators": 350, "reg_lambda": 2.271390116508786, "subsample": 0.7953180584310356, 
fs      : {"strategy": "importance", "n_selected": 30, "selected": ["Time", "V1", "V2", "V3", "V4", "V5", "V6", "V7", "V8", "V9", "V10", "V11", "V12", "V13", "V14", "V15", "V16", "V17", "V18", "

,model_id,score,rank
0,xgboost,0.843642,1
1,catboost,0.839144,2
2,lightgbm,0.833875,3
3,linear,0.761127,4
4,baseline,0.001726,5


## Level 1 vs level 2: what the full pipeline buys

The headline first: **HPO barely moves the needle, and that's the honest
answer**. Level 1 already has every boosting in a healthy PR-AUC range
(catboost 0.84, lightgbm 0.78, xgboost 0.71 — no untuned collapse to rescue).
After 20 Optuna trials the level-2 leaderboard reshuffles the top three into a
near-tie — xgboost 0.8436, catboost 0.8391, lightgbm 0.8339 — with **xgboost**
narrowly winning on OOF PR-AUC.

The shipped result is **essentially level 1, not a regression**: holdout
PR-AUC 0.8825 (level 1) vs 0.8714 (level 2). Feature selection does *not* throw
away any gain — its `auto` cutoff proposes keeping 10 of 30 components, but the
**no-selection gate** judges that subset not significantly better and ships
**all 30 features** instead (`no_selection_gate: no_selection_better`, finding
#10). Caruana ensembling is likewise gated off (`ensemble.applied: false`). The
optimism stays tight (−0.0278), so the report is honest that level 2's extra
machinery buys nothing here — which is exactly how you'd justify shipping the
simpler level-1 model.

One more report detail worth noticing: `hpo.deterministic: false` — a
wall-clock `timeout_s` makes the trial count environment-dependent, and the
run manifest records that honestly (the level-1 run stays byte-reproducible).